## Status (2026-07-31) -- READ BEFORE RUNNING

This is a first-pass, not-yet-executed notebook -- run it top to bottom in Colab and
expect to debug the Metacell-2 cells especially (see the "Run: ..." section below for
known risk points). SuperCell's cells depend only on packages already used elsewhere
in this notebook family (`python-igraph`, `scanpy`, `scikit-learn` -- the latter
already a transitive scanpy dependency, nothing new to install) and should be
lower-risk.

**Purpose.** Reviewer e9Ho (Weakness 7 / Question 5): *"Only SEACells and MetaQ are
true metacell baselines; SuperCell, Metacell-2, and scVI are not run ... Will you add
more baselines like scVI / SuperCell / Metacell-2?"* scVI is already covered as a
correction method in `batch_correct_then_cluster_baselines.ipynb`. This notebook adds
the two still-missing **true metacell baselines**: **SuperCell** and **Metacell-2** --
run directly on each dataset's own within-batch representation, at
`K = DATASETS[ds_id]['num_prototypes']` metacells, exactly like the paper's existing
`SEACells (PCA)` row in Tables 1/2. They are a third and fourth entry in that same
row-for-row comparison, not a new experiment shape.

**SuperCell** (Bilous et al. 2022, *BMC Bioinformatics*, GfellerLab/SuperCell) has no
pip-installable Python port -- checked directly against PyPI, not just GitHub/CRAN:
the `supercell` / `pysupercell` package names that DO exist on PyPI are an unrelated
Tornado REST-API framework and a crystal-structure tool, not this algorithm. The only
Python-reachable path, GfellerLab/MetacellAnalysisToolkit, wraps the R package via
rpy2 + a dedicated conda environment with R installed -- much more fragile to install
in Colab than what's already used here.

`interpretable_ssl/evaluation/extra_metacell_baselines.py::run_supercell_baseline` is
instead a **source-verified Python port** -- read directly from
`R/SCimplify.R` / `R/build_knn_graph.R` on github.com/GfellerLab/SuperCell (not
inferred from the paper abstract) -- matching its exact default recipe: top 1000
variance genes -> z-scored PCA (10 PCs) -> a kNN graph (k=5) that's UNWEIGHTED and
built as a UNION of each cell's neighbor relation (not the adaptive-RBF graph
SEACells(PCA)/Leiden use elsewhere in this codebase) -> `igraph::cluster_walktrap`,
dendrogram cut to exactly K groups. See the module docstring for the full
step-by-step correspondence to the R source. Not a call into the official package --
a faithful, source-checked port of its mechanism, not a guess from the method
description.

**Metacell-2** (Ben-Kiki et al. 2022, *Genome Biology*, tanaylab/metacells) IS a real,
pip-installable package (`pip install metacells`) and is called directly
(`metacells.pl.divide_and_conquer_pipeline`) via `run_metacell2_baseline` in the same
module. It needs raw UMI counts (not log-normalized expression), has no direct
"give me exactly K groups" knob (group count emerges from a `target_metacell_umis`
UMI-budget parameter), and its own outlier/deviant detection can leave some cells
unassigned (`metacell == -1`) -- see that function's docstring for exactly how each
of these is handled. **This is the part most likely to need debugging on first run**
(package install, gene/cell count sanity, `target_metacell_umis` search converging).

Both baselines write `metrics.json` / `*_per_mc.csv` / `cell_assignments.csv` in the
same layout every other baseline in this codebase uses, so they show up automatically
in `load_task1_multi` / `rare_celltype_purity_table` / `show_table` alongside scProto,
SEACells, MetaQ, etc. -- the Results section below reuses those existing helpers
rather than building new display logic.


## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Same base install as batch_correct_then_cluster_baselines.ipynb -- nb_setup.py
# (run further down) unconditionally imports this codebase's FULL eval stack
# (interpretable_ssl.experiments.tasks -> interpretable_ssl.trainers.scproto -> scarches;
# interpretable_ssl.evaluation.paper_figures -> embedding_metrics -> scib_metrics + scvi),
# regardless of what this notebook's own new code (SuperCell/Metacell-2) actually
# touches -- every notebook that runs nb_setup.py needs this same base environment.
# metacells is the one addition specific to this notebook.
!pip install -q scarches faiss-gpu-cu12 scib-metrics
!pip install git+https://github.com/dpeerlab/SEACells.git --quiet --no-deps
!pip install numpy scipy --upgrade -q
!pip install -q palantir harmonypy
!pip install -q python-igraph leidenalg
!pip install -q metacells
!pip install -q "numpy==1.26.4" "scipy==1.13.1"
!pip install --upgrade --force-reinstall numpy cupy-cuda12x
!pip install "numpy<2.3"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.5/187.5 kB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 104.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 106.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 109.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 86.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 128.6 MB/s eta 0:00:00
   ━━━━━

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 99.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.5.1
    Uninstalling numpy-2.5.1:
      Successfully uninstalled numpy-2.5.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
seacells 0.3.3 requires pyranges, which is not installed.
anndata 0.13.2 requires scipy!=1.17.0,>=1.14, but you have scipy 1.13.1 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
access 1.1.10.post3 requires scipy>=1.14.1, but you have scipy 1.13.1 which is incompatible.
pytensor 2.38.3 requires numba<=0.65.1,>=0.58, but you have numba 0.66.0 which is incompatible.
tsfresh 0.21.2 requires scipy>=1.14.0; python_version >= "3.10", but you have scipy 

In [ ]:
# IMPORTANT: restart the runtime after this cell before running the cells below --
# metacells ships compiled C++ extensions; an in-process import after a fresh install
# can pick up a stale/partial build otherwise.


In [ ]:
# Ground truth for "did the install cell above actually work" -- pip's own log is
# noisy (resolver backtracking prints "Getting requirements to build wheel" errors
# for discarded candidate versions even on a fully successful install), so eyeballing
# it is unreliable. Actually importing every package we just installed is the real
# test -- see batch_correct_then_cluster_baselines.ipynb's identical check cell.
_checks = {
    'numpy': 'numpy', 'scipy': 'scipy', 'anndata': 'anndata', 'scanpy': 'scanpy',
    'scarches': 'scarches', 'scvi-tools': 'scvi', 'seacells': 'SEACells',
    'palantir': 'palantir', 'scib-metrics': 'scib_metrics', 'leidenalg': 'leidenalg',
    'python-igraph': 'igraph', 'harmonypy': 'harmonypy', 'faiss-cpu': 'faiss',
    'metacells': 'metacells',
}
_failed = []
for pkg_name, import_name in _checks.items():
    try:
        mod = __import__(import_name)
        ver = getattr(mod, '__version__', '?')
        print(f"  OK   {pkg_name:16s} (import {import_name}, version {ver})")
    except Exception as e:
        _failed.append(pkg_name)
        print(f"  FAIL {pkg_name:16s} (import {import_name}): {type(e).__name__}: {e}")

if _failed:
    print(f"\n{len(_failed)} package(s) failed to import: {_failed} -- re-run that "
          f"package's specific pip install line above and check its full error "
          f"output before proceeding.")
else:
    print(f"\nAll {len(_checks)} packages import cleanly -- safe to continue.")


  OK   numpy            (import numpy, version 2.2.6)
  OK   scipy            (import scipy, version 1.13.1)


/tmp/ipykernel_3795/2030882889.py:17: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  ver = getattr(mod, '__version__', '?')


  OK   anndata          (import anndata, version 0.13.2)


/tmp/ipykernel_3795/2030882889.py:17: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  ver = getattr(mod, '__version__', '?')


  OK   scanpy           (import scanpy, version 1.12.3)


  FAIL scarches         (import scarches): ImportError: cannot import name 'read' from 'anndata' (/usr/local/lib/python3.12/dist-packages/anndata/__init__.py)
  OK   scvi-tools       (import scvi, version 1.5.0.post1)
  OK   seacells         (import SEACells, version 0.3.3)
  OK   palantir         (import palantir, version 1.4.5)
  OK   scib-metrics     (import scib_metrics, version 0.6.0)
  OK   leidenalg        (import leidenalg, version 0.12.0)
  OK   python-igraph    (import igraph, version 1.0.0)
  OK   harmonypy        (import harmonypy, version 2.0.0)
  OK   faiss-cpu        (import faiss, version 1.14.1)
  OK   metacells        (import metacells, version 0.9.5)

1 package(s) failed to import: ['scarches'] -- re-run that package's specific pip install line above and check its full error output before proceeding.


In [16]:
%run /content/drive/MyDrive/codes/interpretable-prototype/notebooks/nb_setup.py


nb_setup done. Available: get_trainer, run_mc_task, fig_*, LAMBDA_PROTO_UMAP, LAMBDA_PROTO_UMAP_PRECON, LAMBDA_PARAM_UMAP, LAMBDA_RECON_ONLY, train_sure, eval_sure_task1/2/3
Configs: {'LAMBDA_PROTO_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'proto'}, 'LAMBDA_PARAM_UMAP': {'lambda_umap': 1, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 0, 'lambda_proto_recon': 0.0, 'umap_similarity': 'embedding'}, 'LAMBDA_RECON_ONLY': {'lambda_umap': 0, 'lambda_swav': 0, 'lambda_kl': 0, 'lambda_recon': 1, 'lambda_proto_recon': 0.0}}


In [17]:
# Extra imports not already covered by nb_setup.py (which already pulls in
# run_mc_task, eval_seacell_task1/2/3, load_task1_multi, show_table,
# rare_celltype_purity_table, TASK1_METRICS, TASK2_METRICS via
# `from interpretable_ssl.evaluation.paper_figures import *` and
# `from interpretable_ssl.evaluation.metric_helpers.result_tables import *`).
from interpretable_ssl.datasets.dataset_configs import DATASETS
from interpretable_ssl.configs.paths import get_dataset_model_dir

from interpretable_ssl.evaluation.extra_metacell_baselines import (
    run_supercell_baseline, run_metacell2_baseline, run_extra_baselines_for_dataset,
)

print("extra imports ready (run_supercell_baseline, run_metacell2_baseline, "
      "run_extra_baselines_for_dataset)")


extra imports ready (run_supercell_baseline, run_metacell2_baseline, run_extra_baselines_for_dataset)


## Config

`K` (target metacell count) defaults to each dataset's `num_prototypes` from
`DATASETS` inside `run_supercell_baseline` / `run_metacell2_baseline` themselves --
matching every other baseline's convention, *"All baselines are configured to produce
the same number of metacells K as scProto."* Nothing to set here for K; this cell only
controls which datasets run and whether to skip already-computed ones.


In [ ]:
RNA_SEQ_DATASETS = ['pancreas', 'lung', 'pbmc-immune']

SKIP_IF_EXISTS = True  # skip a baseline entirely if its metrics.json already exists
                        # on disk -- re-running a dataset cell after a crash/interrupt
                        # never redoes already-saved work. SuperCell's skip check also
                        # verifies K_target still matches (see its docstring); re-runs
                        # automatically if DATASETS[ds_id]['num_prototypes'] changed.

for ds_id in RNA_SEQ_DATASETS:
    print(f"{ds_id}: K = {DATASETS[ds_id]['num_prototypes']}")


pancreas: K = 220
lung: K = 300
pbmc-immune: K = 300


## Run: Pancreas

In [ ]:
pancreas_results = run_extra_baselines_for_dataset('pancreas', skip_if_exists=SKIP_IF_EXISTS)
pancreas_results


[pancreas] supercell already computed -- skipping (metrics.json found, K matches)
[pancreas] metacell2 already computed -- skipping (metrics.json found, realized K=218, target K=220)
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
[pancreas] canonical ARBF-on-PCA affinity graph loaded from ./graphs/affinity_pancreas16382_ncomp50_kneighbors50_arbf.pkl (nnz=1171528) -- caching for reuse across all methods for this dataset.
[/content/drive/MyDrive/models/pancreas/metacell2] modularity recomputed against canonical graph: mean_modularity_batch=0.2172693019846922 +/- 0.10875253480506195 (was 0.2172693019846922), K_target=220


{'supercell': {'mean_cell_type_purity': 0.9027181746068834,
  'weighted_mean_cell_type_purity': 0.9166157978268832,
  'weighted_std_cell_type_purity': 0.14866466115059904,
  'mean_batch_entropy': 0.28343950662561895,
  'weighted_mean_batch_entropy': 0.3937611909802396,
  'weighted_std_batch_entropy': 0.4296718005549073,
  'coverage': 0.8571428571428571,
  'modularity': 0.46020341001556586,
  'n_unused_protos': 0,
  'unused_proto_ratio': 0.0,
  'mean_modularity_batch': 0.3862447982815545,
  'std_modularity_batch': 0.03645480098967325,
  'aff_compactness_mean': 0.5599596339577159,
  'aff_compactness_per_batch': {'celseq': 0.19490369936099788,
   'celseq2': 0.3108674388945423,
   'fluidigmc1': 0.2836898031302109,
   'inDrop1': 0.2867321075692533,
   'inDrop2': 0.23049128541517935,
   'inDrop3': 0.2656227150626521,
   'inDrop4': 0.24306826982964228,
   'smarter': 1.429813925194484,
   'smartseq2': 0.5933700160471966},
  'K_target': 220,
  'scgraph_corr_avg': 0.11514523570742603,
  'scgraph

## Run: Lung

In [ ]:
lung_results = run_extra_baselines_for_dataset('lung', skip_if_exists=SKIP_IF_EXISTS)
lung_results


[lung] supercell already computed -- skipping (metrics.json found, K matches)
[lung] metacell2 already computed -- skipping (metrics.json found, realized K=345, target K=300)
loading lung data
✅ Already subsetted to HVGs (4000 genes).
[lung] canonical ARBF-on-PCA affinity graph loaded from ./graphs/affinity_lung32472_ncomp50_kneighbors50_arbf.pkl (nnz=2480396) -- caching for reuse across all methods for this dataset.
[/content/drive/MyDrive/models/lung/metacell2] modularity recomputed against canonical graph: mean_modularity_batch=0.22724058663421245 +/- 0.05920004381283713 (was 0.22724058663421245), K_target=300


{'supercell': {'mean_cell_type_purity': 0.8986832471931591,
  'weighted_mean_cell_type_purity': 0.8944937176644494,
  'weighted_std_cell_type_purity': 0.15312941749675982,
  'mean_batch_entropy': 0.6818613661061441,
  'weighted_mean_batch_entropy': 0.783757913108141,
  'weighted_std_batch_entropy': 0.49549048123889716,
  'coverage': 1.0,
  'modularity': 0.41283359317526547,
  'n_unused_protos': 0,
  'unused_proto_ratio': 0.0,
  'mean_modularity_batch': 0.3719400512767382,
  'std_modularity_batch': 0.03320787568179805,
  'aff_compactness_mean': 11.13703387708885,
  'aff_compactness_per_batch': {'1': 0.37059819106805286,
   '2': 6.327338093838308,
   '3': 7.842201846288336,
   '4': 17.218077973249013,
   '5': 28.441313417114827,
   '6': 0.054394934579327446,
   'A1': 0.8992593072248638,
   'A2': 0.09284915054525816,
   'A3': 1.5055103901994304,
   'A4': 0.1348129123997896,
   'A5': 1.91371520655522,
   'A6': 3.5697678702987785,
   'B1': 0.5175709075248628,
   'B2': 0.3003370235575875,
  

## Run: PBMC (Immune)

In [ ]:
immune_results = run_extra_baselines_for_dataset('pbmc-immune', skip_if_exists=SKIP_IF_EXISTS)
immune_results


[pbmc-immune] supercell already computed -- skipping (metrics.json found, K matches)
loading pbmc-immune data
✅ Already subsetted to HVGs (4000 genes).
loading pbmc-immune data
✅ Already subsetted to HVGs (4000 genes).


set pbmc-immune_metacell2_iter0.var[lateral_gene]: 0 true (0%) out of 4000 bools
INFO:metacells:set pbmc-immune_metacell2_iter0.var[lateral_gene]: 0 true (0%) out of 4000 bools
set pbmc-immune_metacell2_iter0.var[noisy_gene]: 0 true (0%) out of 4000 bools
INFO:metacells:set pbmc-immune_metacell2_iter0.var[noisy_gene]: 0 true (0%) out of 4000 bools
set pbmc-immune_metacell2_iter0.var[selected_gene]: * -> False
INFO:metacells:set pbmc-immune_metacell2_iter0.var[selected_gene]: * -> False


[pbmc-immune] metacell2 size-search iter 0: target_metacell_umis=1025952 ...


set pbmc-immune_metacell2_iter0.var[rare_gene]: 0 true (0%) out of 4000 bools
INFO:metacells:set pbmc-immune_metacell2_iter0.var[rare_gene]: 0 true (0%) out of 4000 bools
set pbmc-immune_metacell2_iter0.var[rare_gene_module]: 4000 int32 elements with all outliers (100%)
INFO:metacells:set pbmc-immune_metacell2_iter0.var[rare_gene_module]: 4000 int32 elements with all outliers (100%)
set pbmc-immune_metacell2_iter0.obs[cells_rare_gene_module]: 33506 int32 elements with all outliers (100%)
INFO:metacells:set pbmc-immune_metacell2_iter0.obs[cells_rare_gene_module]: 33506 int32 elements with all outliers (100%)
set pbmc-immune_metacell2_iter0.obs[rare_cell]: 0 true (0%) out of 33506 bools
INFO:metacells:set pbmc-immune_metacell2_iter0.obs[rare_cell]: 0 true (0%) out of 33506 bools
set pbmc-immune_metacell2_iter0.var[selected_gene]: 1944 true (48.6%) out of 4000 bools
INFO:metacells:set pbmc-immune_metacell2_iter0.var[selected_gene]: 1944 true (48.6%) out of 4000 bools
set pbmc-immune_metac

  [metacell2] -> 383 metacells + 353 outliers (target 300, tolerance +/-15%)


...

100%|██████████| 549/549 [00:04<00:00, 125.34it/s]


[metacell2] unused protos: 0/549 (0.00%)
[metacell2] mean cell-type purity: 0.9238  (size-weighted: 0.8013 ± 0.1830)
[metacell2] mean batch entropy: 0.2020  (size-weighted: 0.4661 ± 0.4372)
[metacell2] coverage: 0.9375
[metacell2] modularity: 0.2865
[metacell2] per-batch modularity: mean=0.2116, std=0.0808
[aff_dc_compactness] looking for graph at: ./graphs/affinity_pbmc-immune33506_ncomp50_kneighbors50_arbf.pkl
[aff_dc_compactness] mean=3.5227 | saved to /content/drive/MyDrive/models/pbmc-immune/metacell2/aff_dc_compactness.csv
[metacell2] saved metrics to /content/drive/MyDrive/models/pbmc-immune/metacell2
[pbmc-immune] canonical ARBF-on-PCA affinity graph loaded from ./graphs/affinity_pbmc-immune33506_ncomp50_kneighbors50_arbf.pkl (nnz=2624382) -- caching for reuse across all methods for this dataset.
[/content/drive/MyDrive/models/pbmc-immune/metacell2] modularity recomputed against canonical graph: mean_modularity_batch=0.18830731684002264 +/- 0.08594352102273349 (was 0.2116060157

  0%|          | 0/5 [00:00<?, ?it/s]

Deleted: tmp_949d9370.h5ad
[metacell2] umap_cells.csv / umap_protos.csv saved to /content/drive/MyDrive/models/pbmc-immune/metacell2
[pbmc-immune] metacell2 saved to /content/drive/MyDrive/models/pbmc-immune/metacell2


{'supercell': {'mean_cell_type_purity': 0.8882581269476937,
  'weighted_mean_cell_type_purity': 0.8570405300543187,
  'weighted_std_cell_type_purity': 0.14715122326902844,
  'mean_batch_entropy': 0.13898836773026718,
  'weighted_mean_batch_entropy': 0.23431745465762935,
  'weighted_std_batch_entropy': 0.26693431368782305,
  'coverage': 1.0,
  'modularity': 0.31004407302497833,
  'n_unused_protos': 0,
  'unused_proto_ratio': 0.0,
  'mean_modularity_batch': 0.2726538336183254,
  'std_modularity_batch': 0.02967167080848846,
  'aff_compactness_mean': 0.7038517416706815,
  'aff_compactness_per_batch': {'10X': 0.34749441307154366,
   'Freytag': 0.2116710680205597,
   'Oetjen': 0.7246237155254971,
   'Sun': 0.04716096907223105,
   'Villani': 0.30537234639888655},
  'K_target': 300,
  'scgraph_corr_avg': 0.3699828250070503,
  'scgraph_corr_std': 0.20419985186475856,
  'n_clusters_realized': 300},
 'metacell2': {'mean_cell_type_purity': 0.923782237551726,
  'weighted_mean_cell_type_purity': 0.8

## Bonus: SuperCell on scProto's own Stage-1 (corrected) embedding

Optional, off by default. Reviewers F5RB/nG29 separately asked for two-step
baselines (correct-then-cluster) on scProto's own Stage-1 latent -- see
`batch_correct_then_cluster_baselines.ipynb` for the SEACells/Leiden versions of that
comparison. `run_supercell_baseline` is embedding-agnostic (unlike Metacell-2, which
needs real counts and can't be pointed at an 8-dim latent), so the identical
comparison is one call away for SuperCell too: does SuperCell on scProto's own
corrected embedding close the gap, the same question already answered for
SEACells/Leiden. Needs an existing Stage-1 checkpoint for each dataset
(`get_stage1_latent` raises `FileNotFoundError` if none exists yet -- see
`train_scproto_spatial.ipynb` / the main training notebook for how those were produced).


In [ ]:
from interpretable_ssl.evaluation.batch_correct_baselines import get_stage1_latent

stage1_supercell_results = {}
for ds_id in RNA_SEQ_DATASETS:
    t, ad, z1 = get_stage1_latent(ds_id)
    ad.obsm['X_stage1z'] = z1
    stage1_supercell_results[ds_id] = run_supercell_baseline(
        ds_id, latent_key='X_stage1z', skip_if_exists=SKIP_IF_EXISTS,
    )

stage1_supercell_results


 captum (see https://github.com/pytorch/captum).


dataset is None, loading pancreas
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).
stage1_latent_extract_ds-panc_NP220_aff-arbf_cvae_e50_v31
Embedding dictionary:
 	Num conditions: [9]
 	Embedding dim: [10]
Encoder Architecture:
	Input Layer in, out and cond: 4000 64 10
	Mean/Var Layer in/out: 64 8
Decoder Architecture:
	First Layer in, out and cond:  8 64 10
	Output Layer in/out:  64 4000 

📊 Affinity: wdeg[min/mean/max]=5.446/23.916/92.201, effk_med=62.4, mutual=100.00%
adam
Loaded pretrain checkpoint from /content/drive/MyDrive/models/pancreas/pretrain/pretrain_ds-pancreas_cvae_e50/pretrain_checkpoint.pth
  pretrain_params: {'dataset_id': 'pancreas', 'cvae_epochs': 50, 'batch_size': 1024, 'latent_dims': 8, 'l2norm': 1, 'model_type': 'gm', 'beta': 0.3, 'condition_key': 'tech'}


  0%|          | 0/16 [00:00<?, ?it/s]

[pancreas] Stage-1 latent: 16382 cells x 8 dims
loading pancreas data
✅ Already subsetted to HVGs (4000 genes).


KeyError: "ad.obsm['X_stage1z'] not found -- compute/attach it before calling run_supercell_baseline(latent_key='X_stage1z')."

## Results comparison

Pulls in **every** existing run under `MODEL_DIR/{ds}/*` (scProto, SEACells(PCA),
MetaQ, scPoli-cVAE, UMAP, the batch-correct-then-cluster baselines, ...) alongside
`supercell` / `metacell2` (and `supercell_X_stage1z` if the bonus cell above was run)
computed here -- `load_task1_multi` / `rare_celltype_purity_table` just scan folders +
`metrics.json` / `umap_cells.csv`, they don't need to know these runs are new.

**Reminder:** modularity below is always scored against the canonical ARBF-on-PCA
graph (`recompute_modularity_canonical`, called inside both baseline functions), never
against SuperCell's own kNN graph or whatever internal graph Metacell-2 built -- same
non-circularity guarantee every other baseline in this codebase gets.


In [ ]:
dataset_display_names = {'pancreas': 'Pancreas', 'lung': 'Lung', 'pbmc-immune': 'Immune'}

# Keyword -> display name. 'seacell' resolves to get_seacell_model_dir(ds_id) directly
# (see _resolve_run_dir) -- the paper's existing within-batch 'SEACells (PCA)' row,
# the natural reference point for these two new within-batch baselines.
MODEL_KEYWORDS = {
    'seacell': 'SEACells (PCA)',
    'metaq': 'MetaQ',
    'supercell': 'SuperCell',
    'metacell2': 'Metacell-2',
}
# Uncomment if the bonus Stage-1-embedding cell above was run:
# MODEL_KEYWORDS['supercell_X_stage1z'] = 'SuperCell (Stage-1 emb.)'

# scProto's own canonical run folders (see batch_correct_then_cluster_baselines.ipynb's
# Table-1 cell for why this exact-match-then-normalize approach is needed rather than
# a plain keyword substring match).
SCPROTO_CANONICAL_RUNS = {
    'proto_umap_ds-panc_NP220_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
    'proto_umap_ds-lung_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
    'proto_umap_prtInit-wayp_aff-arbf_lprec0.01_usim-prot_cvae_e50_ecal1_lpu0.1_pum-ema_lna1_nagg-max_upm-dotp_v31',
}
SCPROTO_KEY = extract_model_key(next(iter(SCPROTO_CANONICAL_RUNS)))
assert all(extract_model_key(r) == SCPROTO_KEY for r in SCPROTO_CANONICAL_RUNS)

def _keep_and_rename_runs(df):
    keep_keys = {extract_model_key(r): name for r, name in MODEL_KEYWORDS.items()}
    keep_keys[SCPROTO_KEY] = 'scProto'
    idx_keys = df.index.get_level_values('run')
    mask = idx_keys.isin(keep_keys.keys())
    out = df[mask].copy()
    new_run = [keep_keys[k] for k in out.index.get_level_values('run')]
    out.index = pd.MultiIndex.from_arrays(
        [out.index.get_level_values('dataset'), new_run], names=['dataset', 'run'],
    )
    out = out[~out.index.duplicated(keep='first')]
    return out


### Completeness check: which baselines actually finished, per dataset

Run this before reading the tables below -- a method missing from a dataset just
shows up as blank/NaN in Table 1/2 further down, easy to miss. This scans
`MODEL_DIR/{ds}/*` directly (same folders `load_task1_multi` reads) and reports a
per-(dataset, method) present/missing matrix for every method this notebook's tables
expect, including scProto and the pre-existing baselines, not just SuperCell/Metacell-2.


In [18]:
df_present = load_task1_multi(RNA_SEQ_DATASETS)

expected = dict(MODEL_KEYWORDS)  # {keyword: display_name} -- seacell/metaq/supercell/metacell2
expected_scproto_key = SCPROTO_KEY  # already computed in the cell above

rows = []
for ds_id in RNA_SEQ_DATASETS:
    ds_runs = (
        set(df_present.loc[ds_id].index)
        if (not df_present.empty and ds_id in df_present.index.get_level_values('dataset'))
        else set()
    )
    row = {'dataset': dataset_display_names.get(ds_id, ds_id)}
    row['scProto'] = any(extract_model_key(r) == expected_scproto_key for r in ds_runs)
    for kw, name in expected.items():
        row[name] = any(kw in extract_model_key(r) for r in ds_runs)
    rows.append(row)

df_completeness = pd.DataFrame(rows).set_index('dataset')
display(df_completeness)

missing = [
    (ds, method)
    for ds, row in df_completeness.iterrows()
    for method, ok in row.items()
    if not ok
]
if missing:
    print(f"\nMISSING ({len(missing)}): " + ", ".join(f"{ds}/{method}" for ds, method in missing))
else:
    print("\nAll expected methods present for all datasets.")


,scProto,SEACells (PCA),MetaQ,SuperCell,Metacell-2
dataset,,,,,
Pancreas,True,True,True,True,True
Lung,True,True,True,True,True
Immune,True,True,True,True,True



All expected methods present for all datasets.


### Table 1 (metacell quality): purity, batch entropy, modularity, coverage

In [19]:
df_task1 = load_task1_multi(RNA_SEQ_DATASETS, metrics=TASK1_METRICS)
df_task1 = _keep_and_rename_runs(df_task1)
show_table(df_task1, metrics=TASK1_METRICS, dataset_display_names=dataset_display_names)


### Table 2 (metacell representation quality): coverage, scGraph

In [20]:
df_task2 = load_task1_multi(RNA_SEQ_DATASETS, metrics=TASK2_METRICS)
df_task2 = _keep_and_rename_runs(df_task2)
show_table(df_task2, metrics=TASK2_METRICS, dataset_display_names=dataset_display_names)


### Rare-cell-type table (the key hypothesis test): coverage + homogeneity + F1

Same per-batch-rare-cell definition `results.tex` uses for Table 2 (`tab:rare_cells`).


In [21]:
df_rare = rare_celltype_purity_table(RNA_SEQ_DATASETS, MODEL_KEYWORDS | {r: 'scProto' for r in SCPROTO_CANONICAL_RUNS})
df_rare = df_rare[~df_rare.index.duplicated(keep='first')]

# FIXED 2026-07-31: the metric names below were wrong (rare_ct_coverage / rare_ct_repr_mean /
# _batch_rare_f1_macro_mean with a leading underscore don't exist -- _rare_table_one actually
# returns batch_rare_coverage_mean, batch_rare_repr_macro_mean, batch_rare_homogeneity_mean,
# batch_rare_f1_macro_mean, no leading underscore -- see its return dict in paper_figures.py).
# The wrong names silently rendered a completely empty table (no crash) instead of erroring --
# this is the corrected version. _n_metacells is included so each method's realized metacell
# count (K) sits right next to its scores -- for Metacell-2 this includes singleton
# pseudo-metacells created for its own outlier cells (see run_metacell2_baseline's docstring),
# so its _n_metacells is NOT directly comparable to the other methods' K -- shown here anyway,
# unfiltered, so the actual numbers are visible regardless of that mismatch (the separate
# significance-test cell below is what applies the strict same-K filter, not this table).
show_table(
    df_rare, dataset_display_names=dataset_display_names,
    metrics=['batch_rare_coverage_mean', 'batch_rare_coverage_std',
             'batch_rare_repr_macro_mean', 'batch_rare_repr_macro_std',
             'batch_rare_homogeneity_mean', 'batch_rare_homogeneity_std',
             'batch_rare_f1_macro_mean', 'batch_rare_f1_macro_std',
             '_n_metacells'],
)


  [SEACells (PCA)|pancreas] resolving run dir ...
  [MetaQ|pancreas] resolving run dir ...
  [SuperCell|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] run dir resolved (0.0s)
  [Metacell-2|pancreas] resolving run dir ...
  [scProto|pancreas] resolving run dir ...
  [scProto|pancreas] resolving run dir ...
  [scProto|pancreas] resolving run dir ...
  [SEACells (PCA)|pancreas] reading umap_cells.csv ...
  [SEACells (PCA)|lung] resolving run dir ...
  [SEACells (PCA)|lung] run dir resolved (0.0s)
  [SEACells (PCA)|lung] reading umap_cells.csv ...
  [SEACells (PCA)|pancreas] umap_cells.csv loaded (16382 rows, 0.0s)
  [SEACells (PCA)|pancreas] reading umap_protos.csv ...
  [SEACells (PCA)|pancreas] umap_protos.csv loaded (0.0s)  [SEACells (PCA)|lung] umap_cells.csv loaded (32472 rows, 0.0s)

  [SEACells (PCA)|lung] reading umap_protos.csv ...
  [SEACells (PCA)|pancreas] batch='tech' | 9 unique values, e.g. ['celseq', 'celseq2', 'fluidigmc1', 'inDrop1', 'inDrop2']
  [SEACells (P

### Significance test: scProto vs. SuperCell / Metacell-2 on the rare-cell metrics

Paired one-sided Wilcoxon signed-rank (H1: scProto > other), Bonferroni-corrected --
same test already used for every other baseline's rare-cell comparison in
`batch_correct_then_cluster_baselines.ipynb`. Only same-K comparisons are made (see
that notebook's equivalent cell) -- Metacell-2's realized K is approximate
(`n_clusters_realized` in its `metrics.json`, target-vs-realized within
`size_tol`), so check the printed K's below before reading too much into an 'ns' result.


In [22]:
df_sig_paired = rare_metric_significance_paired(
    df_rare,
    ref_name='scProto',
    metrics=(
        '_batch_rare_f1_macro_per_batch',
        '_batch_rare_homogeneity_per_batch',
    ),
    dataset_display_names=dataset_display_names,
)
df_sig_paired


Skipped (K mismatch vs reference -- not a same-K comparison, per the paper's baseline protocol):
  - Pancreas/batch_rare_f1_macro: Metacell-2 skipped (K=426 vs scProto's K=219, outside 5% tolerance)
  - Pancreas/batch_rare_homogeneity: Metacell-2 skipped (K=426 vs scProto's K=219, outside 5% tolerance)
  - Lung/batch_rare_f1_macro: Metacell-2 skipped (K=346 vs scProto's K=298, outside 5% tolerance)
  - Lung/batch_rare_homogeneity: Metacell-2 skipped (K=346 vs scProto's K=298, outside 5% tolerance)
  - Immune/batch_rare_f1_macro: Metacell-2 skipped (K=549 vs scProto's K=294, outside 5% tolerance)
  - Immune/batch_rare_homogeneity: Metacell-2 skipped (K=549 vs scProto's K=294, outside 5% tolerance)


,dataset,metric,method,k,n,median,mean,std,n_wins,p_vs_ref,p_adj,sig
0,Pancreas,batch_rare_f1_macro,SEACells (PCA),220,8,0.372628,0.370589,0.250869,6.0,0.039062,0.117188,ns
1,Pancreas,batch_rare_f1_macro,MetaQ,217,8,0.068861,0.109538,0.136172,8.0,0.003906,0.011719,*
2,Pancreas,batch_rare_f1_macro,SuperCell,220,8,0.000000,0.074406,0.111247,8.0,0.003906,0.011719,*
3,Pancreas,batch_rare_f1_macro,scProto,219,8,0.437680,0.535403,0.207961,NaN,NaN,NaN,NaN
4,Pancreas,batch_rare_homogeneity,SEACells (PCA),220,8,0.369265,0.420592,0.195138,8.0,0.003906,0.011719,*
5,Pancreas,batch_rare_homogeneity,MetaQ,217,8,0.174473,0.200672,0.116211,8.0,0.003906,0.011719,*
6,Pancreas,batch_rare_homogeneity,SuperCell,220,8,0.163410,0.143800,0.062260,8.0,0.003906,0.011719,*
7,Pancreas,batch_rare_homogeneity,scProto,219,8,0.522129,0.564854,0.155263,NaN,NaN,NaN,NaN
8,Lung,batch_rare_f1_macro,SEACells (PCA),300,15,0.498294,0.420330,0.217382,11.0,0.001678,0.005035,**
9,Lung,batch_rare_f1_macro,MetaQ,292,15,0.626944,0.583566,0.225045,7.0,0.619232,1.000000,ns


### Realized K vs. target K (sanity check)

SuperCell's dendrogram cut hits K exactly by construction (`n_clusters_realized`
should equal `DATASETS[ds_id]['num_prototypes']` every time -- flag it if not, that
would mean the kNN graph had fewer distinct components/leaves than K, extremely
unlikely at these dataset sizes). Metacell-2's is only approximate -- this is the
place to check how close the `target_metacell_umis` search actually landed.


In [23]:
rows = []
for ds_id in RNA_SEQ_DATASETS:
    target_k = DATASETS[ds_id]['num_prototypes']
    for tag in ['supercell', 'metacell2']:
        path = os.path.join(get_dataset_model_dir(ds_id), tag, 'metrics.json')
        if not os.path.exists(path):
            continue
        m = json.load(open(path))
        rows.append({
            'dataset': ds_id, 'method': tag, 'target_k': target_k,
            'n_clusters_realized': m.get('n_clusters_realized'),
            'n_outliers': m.get('n_outliers'),
            'target_metacell_umis': m.get('target_metacell_umis'),
        })

pd.DataFrame(rows)


,dataset,method,target_k,n_clusters_realized,n_outliers,target_metacell_umis
0,pancreas,supercell,220,220,NaN,NaN
1,pancreas,metacell2,220,218,208.0,3.985277e+06
2,lung,supercell,300,300,NaN,NaN
3,lung,metacell2,300,345,1.0,8.830548e+04
4,pbmc-immune,supercell,300,300,NaN,NaN
5,pbmc-immune,metacell2,300,277,272.0,1.309799e+06
